# 샘플 데이터 수집 - 도서관정보나루 API

In [1]:
import requests
import json
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv("../../.env")

LOAN_ITEM_SRCH_URL = "https://data4library.kr/api/loanItemSrch"
AUTH_KEY = os.getenv("LIBRARY_API_KEY")

print("API KEY 확인:", AUTH_KEY[:8] + "..." if AUTH_KEY else "❌ 키 없음")

API KEY 확인: 16424b20...


## 1. 단건 요청 테스트

In [2]:
end_dt = datetime.today().strftime("%Y-%m-%d")
start_dt = (datetime.today() - timedelta(days=30)).strftime("%Y-%m-%d")

params = {
    "authKey": AUTH_KEY,
    "startDt": start_dt,
    "endDt": end_dt,
    "pageNo": 1,
    "pageSize": 5,
    "format": "json",
}

response = requests.get(LOAN_ITEM_SRCH_URL, params=params)
response.raise_for_status()
data = response.json()

# 응답 구조 확인
print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "response": {
    "request": {
      "startDt": "2026-04-14",
      "endDt": "2026-05-14",
      "pageNo": 1,
      "pageSize": 5,
      "format": "json"
    },
    "resultNum": 5,
    "numFound": 5000,
    "docs": [
      {
        "doc": {
          "no": 1,
          "ranking": "1",
          "bookname": "소년이 온다 :한강 장편소설 ",
          "authors": "지은이: 한강",
          "publisher": "창비",
          "publication_year": "2014",
          "isbn13": "9788936434120",
          "addition_symbol": "03810",
          "vol": "",
          "class_no": "813.62",
          "class_nm": "문학 > 한국문학 > 소설",
          "bookImageURL": "http://image.aladin.co.kr/product/4086/97/cover/8936434128_2.jpg",
          "bookDtlUrl": "https://data4library.kr/bookV?seq=2037399",
          "loan_count": "3667"
        }
      },
      {
        "doc": {
          "no": 2,
          "ranking": "2",
          "bookname": "안녕이라 그랬어 :김애란 소설 ",
          "authors": "지은이: 김애란",
          "publisher": "문학동네",
          

## 2. 파싱 확인

In [3]:
docs = data.get("response", {}).get("docs", [])

books = []
for item in docs:
    doc = item.get("doc", {})
    books.append({
        "title": doc.get("bookname", ""),
        "author": doc.get("authors", ""),
        "publisher": doc.get("publisher", ""),
        "publish_year": doc.get("publication_year", ""),
        "isbn": doc.get("isbn13", ""),
        "genre": doc.get("class_nm", ""),
        "loan_count": int(doc.get("loan_count", 0)),
        "image_url": doc.get("bookImageURL", ""),
    })

for book in books:
    print(book)

{'title': '소년이 온다 :한강 장편소설 ', 'author': '지은이: 한강', 'publisher': '창비', 'publish_year': '2014', 'isbn': '9788936434120', 'genre': '문학 > 한국문학 > 소설', 'loan_count': 3667, 'image_url': 'http://image.aladin.co.kr/product/4086/97/cover/8936434128_2.jpg'}
{'title': '안녕이라 그랬어 :김애란 소설 ', 'author': '지은이: 김애란', 'publisher': '문학동네', 'publish_year': '2025', 'isbn': '9791141602376', 'genre': '문학 > 한국문학 > 소설', 'loan_count': 3456, 'image_url': 'https://image.aladin.co.kr/product/36566/52/cover200/k462039240_1.jpg'}
{'title': '작별하지 않는다 :한강 장편소설 ', 'author': '지은이: 한강', 'publisher': '문학동네', 'publish_year': '2021', 'isbn': '9788954682152', 'genre': '문학 > 한국문학 > 소설', 'loan_count': 3111, 'image_url': 'https://image.aladin.co.kr/product/27877/5/cover/8954682154_1.jpg'}
{'title': '혼모노 :성해나 소설집 ', 'author': '지은이: 성해나', 'publisher': '창비', 'publish_year': '2025', 'isbn': '9788936439743', 'genre': '문학 > 한국문학 > 소설', 'loan_count': 3015, 'image_url': 'https://image.aladin.co.kr/product/36101/66/cover200/893643974x_1.j

## 3. 알라딘 API 연동 테스트

In [4]:
import xml.etree.ElementTree as ET
import re
import time

ALADIN_SEARCH_URL = "http://www.aladin.co.kr/ttb/api/ItemSearch.aspx"
ALADIN_API_KEY = os.getenv("ALADIN_API_KEY")

print("알라딘 API KEY 확인:", ALADIN_API_KEY[:8] + "..." if ALADIN_API_KEY else "❌ 키 없음")


def clean_title(title):
    return re.split(r"\s*:", title)[0].strip()


def extract_author_name(author_str):
    if ":" in author_str:
        return author_str.split(":")[1].strip()
    return author_str.strip()


def search_aladin(title):
    params = {
        "ttbkey": ALADIN_API_KEY,
        "Query": clean_title(title),
        "QueryType": "Title",
        "MaxResults": 10,
        "start": 1,
        "SearchTarget": "Book",
        "output": "xml",
        "Version": "20131101",
    }
    response = requests.get(ALADIN_SEARCH_URL, params=params)
    response.raise_for_status()
    return response.text


# 단건 테스트 + 디버그
test_book = books[0]
print(f"\n제목: {test_book['title']}")
print(f"저자: {test_book['author']}")

xml_text = search_aladin(test_book["title"])

# 실제 XML 응답 확인
print("\n--- 원본 XML (처음 2000자) ---")
print(xml_text[:2000])
print("--- 끝 ---")

# 파싱 구조 확인
root = ET.fromstring(xml_text)
print(f"\nroot 태그: {root.tag}")
print(f"직계 자식 태그들: {[child.tag for child in root][:10]}")
items = root.findall("item")
print(f"\nitem 개수: {len(items)}")
if items:
    print(f"첫 번째 item의 자식 태그들: {[child.tag for child in items[0]]}")
    print(f"author: {items[0].findtext('author', '')}")
    print(f"description: {items[0].findtext('description', '')[:200]}")

알라딘 API KEY 확인: ttbjeong...

제목: 소년이 온다 :한강 장편소설 
저자: 지은이: 한강

--- 원본 XML (처음 2000자) ---
<?xml version="1.0" encoding="UTF-8"?>
<object xmlns="http://www.aladin.co.kr/ttb/apiguide.aspx">
    <title>알라딘 검색결과 - 소년이 온다</title>
    <link>http://www.aladin.co.kr/search/wsearchresult.aspx?KeyTitle=%bc%d2%b3%e2%c0%cc+%bf%c2%b4%d9&amp;SearchTarget=book&amp;partner=openAPI</link>
    <logo>http://image.aladin.co.kr/img/header/2011/aladin_logo_new.gif</logo>    
	<pubDate>Thu, 14 May 2026 07:08:11 GMT</pubDate>
	<totalResults>17</totalResults>
    <startIndex>1</startIndex>
    <itemsPerPage>10</itemsPerPage>
    <query>소년이 온다</query>
    <version>20131101</version>    
    <searchCategoryId>0</searchCategoryId>
    <searchCategoryName>전체</searchCategoryName>
    
			<item itemId="40869703">
			    <title>소년이 온다 - 2024 노벨문학상 수상작가</title>
				<link>https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=40869703&amp;partner=openAPI&amp;start=api</link>
				<author>한강 (지은이)</author>
				<pubDate>2014-0

## 4. 전체 수집 + 알라딘 description 보강

In [5]:
from sample_collector import SampleCollector, AladinEnricher

collector = SampleCollector()
books = collector.collect(total=500)    # 2000개 데이터

enricher = AladinEnricher(delay=0.5)
enriched_books = enricher.enrich(books)

collector.save(enriched_books, path="../../data/raw/sample_books.json")

✅ 500건 수집 완료 (2026-04-14 ~ 2026-05-14)
진행: 50/500 | description 확보: 38건
진행: 100/500 | description 확보: 77건
진행: 150/500 | description 확보: 106건
진행: 200/500 | description 확보: 134건
진행: 250/500 | description 확보: 165건
진행: 300/500 | description 확보: 204건
진행: 350/500 | description 확보: 243건
진행: 400/500 | description 확보: 282건
진행: 450/500 | description 확보: 314건
진행: 500/500 | description 확보: 344건
✅ 저장 완료 → ../../data/raw/sample_books.json
